# Donut fine-tune — realistic InBody **270** v2 (issue #13) · Kaggle runner

Retrains Donut on the **new realistic 270** synthetic sheets to close the synthetic→real domain gap.
Fresh from `naver-clova-ix/donut-base`, **1 epoch**, **270-only** (570 deferred).

**Kaggle setup before running:**
1. *Settings → Accelerator* → **GPU T4 x2** (we pin to one GPU below — donut-base OOMs on GPU0 with both).
2. *Settings → Internet* → **On**.
3. *Add-ons → Secrets* → add **`GH_TOKEN`** (fine-grained, Contents: Read-only, scoped to QeekOw/CERA).
4. Upload `synth_270_v2.zip` to Google Drive, share it *anyone-with-link*, and paste its **file id** into `DATA_FILE_ID` below.

For an unattended run use **Save Version → Save & Run All (Commit)** — it runs headless (~90 min) and the checkpoint lands in the version output.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Clone the issue-#13 branch (private repo -> GH_TOKEN secret) and install.
BRANCH = 'feat/realistic-synthetic-270'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/CERA.git'
except Exception:
    REPO = 'https://github.com/QeekOw/CERA.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown


In [ ]:
# Pull the new 270 dataset zip from Drive and unpack to /kaggle/tmp (NOT
# /kaggle/working, so the 2500 images don't bloat the saved version output).
DATA_FILE_ID = '<<PASTE_DRIVE_FILE_ID_FOR_synth_270_v2.zip>>'
DATA_DIR = '/kaggle/tmp/synth_270_v2'
import os, shutil
os.makedirs('/kaggle/tmp', exist_ok=True)
!gdown --id $DATA_FILE_ID -O /kaggle/tmp/synth_270_v2.zip
shutil.unpack_archive('/kaggle/tmp/synth_270_v2.zip', '/kaggle/tmp')
# zip contains the synth_270_v2/ folder; confirm the pngs are where we expect
import glob
n = len(glob.glob(DATA_DIR + '/*.png'))
print('sheets found:', n)
assert n > 0, f'no pngs under {DATA_DIR} — check the zip layout / DATA_FILE_ID'


In [ ]:
# Fresh from donut-base, 1 epoch (memory: more epochs neither help nor hurt on
# this synthetic — 1 epoch for cost). Old checkpoint kept only as the 'before'
# baseline; we do NOT resume from it. batch 1 + grad-accum 4 fits the 2560x1920
# canvas on a T4; effective batch 4.
CHECKPOINT_DIR = '/kaggle/working/donut-270-v2'
!python -m cera.training.train \
  --data-dir /kaggle/tmp/synth_270_v2 --output-dir $CHECKPOINT_DIR \
  --model-name-or-path naver-clova-ix/donut-base \
  --epochs 1 --batch-size 1 --gradient-accumulation-steps 4 --learning-rate 3e-5


In [ ]:
# The checkpoint (model.safetensors + full processor) is under /kaggle/working,
# so it is included in the Save Version output — download it from the version's
# Output tab, then place it at D:/cera/checkpoints/donut-270-v2 for the local
# real-photo + held-out eval (cera.compare --skip-vlm).
!ls -la /kaggle/working/donut-270-v2


## After the run
- **Held-out synthetic:** `python -m cera.compare --data-dir <holdout_270_v2> --donut-checkpoint donut-270-v2 --skip-vlm`
- **Real-photo test:** same command against `D:/cera/data/real_holdout` — does it clear the old **0%**?
- Because we run exactly 1 epoch **to completion**, `train.py` saves the processor itself — no manual processor-attach needed (that gotcha was only for early-interrupt runs).
